In [0]:
from pyspark.sql import functions as F

# Configure ADLS access
storage_key = dbutils.secrets.get(scope="azure-storage", key="storage-account-key")
spark.conf.set(
    "fs.azure.account.key.avddevstorageacc2026.dfs.core.windows.net",
    storage_key
)
ADLS_ROOT = "abfss://datalakecontainer@avddevstorageacc2026.dfs.core.windows.net"

# Read Silver Delta table
df_silver = spark.read.format("delta").load(f"{ADLS_ROOT}/silver/sales")

# Aggregate KPIs
df_gold = df_silver.groupBy(
    "region", "city", "category", "payment_mode"
).agg(
    F.count("sale_id").alias("total_orders"),
    F.sum("amount").alias("total_revenue"),
    F.avg("amount").alias("avg_order_value"),
    F.sum("quantity").alias("total_units_sold"),
    F.countDistinct("store_id").alias("active_stores"),
    F.round(
    F.sum(F.when(F.col("payment_mode") == "UPI", F.col("amount"))) / F.sum("amount") * 100, 2
    ).alias("upi_revenue_pct")
).withColumn("created_at", F.current_timestamp())

# Write Gold as Delta in ADLS
gold_path = f"{ADLS_ROOT}/gold/sales_kpis"
df_gold.write.format("delta").mode("overwrite").save(gold_path)

print(f"Gold: {df_gold.count()} KPI rows written")

In [0]:
spark.read.format("delta").load(gold_path).display()